# H20 匹配预测与持有期限：输入资格已检查

本 notebook 目前执行原生 Qlib 输入检查：不拟合、不回测、不下单。固定实验的模型接口仍在实现；not_promoted。


## 修订记录（四项待核）
1. `registered_at_utc` 用真实当前 UTC draft 时间；`technical_timestamp_correction` 记录这是技术字段更正，**不假装实验更早冻结**。
2. provider 使用**真实 content hash**（calendars/instruments/features + manifest，排除 qlib.yaml/cache）+ 价格/日历 readback + H5 reference R 对象 full SHA，已冻结，不留待将来计算。
3. H5 reference **唯一一条**：`benchmark_terminal_value_20261009[f79e]` benchmark_terminal_H5（annual fits/forecasts 源自 purged `d467`）；H20 用**同一 benchmark-terminal objective/daily-risk accounting**，只改 20 label + planning + hold；`selected_original` 仅背景、不作候选；不做原始终值H5与bench-terminalH20混用。
4. gates 从父 plan **完整复制数值定义**（两成本 positive log excess、6/7 正年、annual median、leave-one-year-out、rolling252≥60%、20/60 block CI lower>0、MaxDD/CVaR 不劣 SOXX、1e-8 下限），删除 `historical_gate_expected`，结果必须实算。

## 因果/时点合同
- features `<= signal_t 16:15`；`prediction_at = signal_t 16:15`。
- `label_h20_available_at = t+21 真实 D.calendar open 09:31 NY`（非自然日 +1）。
- `allocation_decision_at` 在 **next-open 捕获 pretrade quote 之后**（≥ 次日 09:31 NY）；**不得**伪称前日 16:15。
- Sigma 用已知 previous-close 估计，但约束在**实际开盘 target** 上评估；variance 约束时点 = `allocation_decision_at`。
- 历史 9:30 open 仅 explicit proxy；paper 用捕获 quote。
- **source_compatibility_audit_only**：H5 legacy 16:15 与 next-open pretrade 的 clock/依赖错配仅审计、不改封存父、不洗绿。

## 单位
- `mu20 = twenty_session_simple_return`（**非 squared**）
- `Vmean20 = twenty_session_simple_return_squared`
- `Sigma_daily = one_session_simple_return_squared`

## 来源 / 参考绑定
- price/features: `data/qlib/published_futu_affine_20260925_v1`（content `b46e8edf…`）；execution: `data/qlib/legacy_soxl_execution_20260925_v1`（content `fe475a1c…`）；2016-01-04 → 2026-09-25。
- H5 reference：`benchmark_terminal_value_20261009`（Recorder `f79e065bfc4e4ec7996da5f1a5379e16`，variant `benchmark_terminal`）；annual fits/forecasts 源自 `purged_ridge_regularization_20261009`（`d46772566a2947859faee38a5d4e08b8`）。
- **不混** source17（17f81fbf, uniform ~2690）/uniform，不换 pricing。
- 2692 只是 H5 父样本成熟数；**H20 成熟数须从真实价格/日历/截止另算**。

In [1]:
import sys
from pathlib import Path
_workspace = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "SPEC.md").exists())
sys.path.insert(0, str(_workspace))
from pathlib import Path
import json
import qlib
from qlib.data import D
from qlib.workflow import R
from src.analysis.h20_prefit_contract import run_prefit_contract

root = Path.cwd().resolve()
while not (root / "SPEC.md").exists():
    if root == root.parent:
        raise RuntimeError("workspace root unavailable")
    root = root.parent
card = root / "research" / "h20_matched_forecast_execution_20261010"
plan = json.loads((card / "preregister_plan.json").read_text())
qlib.init(provider_uri=str(root / plan["source_binding"]["price_features_provider"]), region="us", kernels=1)
summary = run_prefit_contract(root=root, card_dir=card)
print(json.dumps({k: summary[k] for k in ("status", "recorder_id", "recorder_status", "horizons", "native_training_dataset")}, ensure_ascii=False, indent=2))


2026/10/11 12:21:29 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[56172:MainThread](2026-10-11 12:21:30,344) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[56172:MainThread](2026-10-11 12:21:30,346) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[56172:MainThread](2026-10-11 12:21:30,346) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


...

[56172:MainThread](2026-10-11 12:21:30,640) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


[56172:MainThread](2026-10-11 12:21:30,642) INFO - qlib.workflow - [exp.py:258] - Experiment 129083020028550798 starts running ...


[56172:MainThread](2026-10-11 12:21:30,719) INFO - qlib.workflow - [recorder.py:345] - Recorder 5654796769704dee8d84d51186ee4960 starts running under Experiment 129083020028550798 ...


[56172:MainThread](2026-10-11 12:21:31,570) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{
  "status": "prefit_input_contract_pass_model_integration_pending",
  "recorder_id": "5654796769704dee8d84d51186ee4960",
  "recorder_status": "FINISHED",
  "horizons": {
    "5": {
      "rows": 2698,
      "terminal_unknown": 6,
      "mature_rows": 2692,
      "last_mature_signal": "2026-09-17",
      "label_provider": "data/qlib/legacy_soxl_execution_20260925_v1",
      "mean_unit": "five_session_simple_return",
      "availability_clock": "09:31 America/New_York"
    },
    "20": {
      "rows": 2698,
      "terminal_unknown": 21,
      "mature_rows": 2677,
      "last_mature_signal": "2026-08-26",
      "label_provider": "data/qlib/legacy_soxl_execution_20260925_v1",
      "mean_unit": "twenty_session_simple_return",
      "availability_clock": "09:31 America/New_York"
    }
  },
  "native_training_dataset": {
    "class_name": "DatasetH",
    "rows": 3956,
    "features": 25,
    "labels": [
      "soxl_return_h20",
      "soxx_return_h20"
    ],
    "first_signal": "2011-01-03

In [2]:
from IPython.display import Markdown, display
display(Markdown((card / "conclusion.md").read_text()))


# H20 输入资格检查

实际 Qlib 双来源读回 2698 个评估交易日，哈希与冻结计划一致且未变化。H20 成熟标签 2677 条，末尾 21 条保持未知。

实际 DatasetH 有 3956 条记录、25 个特征、两个 H20 标签，从 2011-01-03 开始，错误 horizon 被阻断。这仅验证输入，未训练、未回测，无收益或 regime/window 投资结论。

决定：not_promoted。H5 经济影响尚未核验，下一步完成原生模型和选择接口，再执行固定 H20 实验。Recorder `5654796769704dee8d84d51186ee4960`，状态 `FINISHED`。


# H20 输入资格检查

实际 Qlib 双来源读回 2698 个评估交易日，哈希与冻结计划一致且未变化。H20 成熟标签 2677 条，末尾 21 条保持未知。

实际 DatasetH 有 3956 条记录、25 个特征、两个 H20 标签，从 2011-01-03 开始，错误 horizon 被阻断。这仅验证输入，未训练、未回测，无收益或 regime/window 投资结论。

决定：not_promoted。H5 经济影响尚未核验，下一步完成原生模型和选择接口，再执行固定 H20 实验。Recorder `5654796769704dee8d84d51186ee4960`，状态 `FINISHED`。
